# 일별 가격정보 일일 자동 수집 (Daily Job)

매일 스케줄러(Databricks Jobs)에 의해 실행되며, **실행일 당일의 일별 가격정보**를 공공데이터 API에서 호출하여 저장합니다.

### 구조
- `api_client.py`의 `ATAPIClient.fetch_day_price()`를 사용합니다.
- `ref_sheet_품목코드.csv`를 참조하여 전체 품목을 순회합니다.
- 품목별 페이지네이션으로 모든 데이터를 수집합니다.
- 수집 결과를 CSV 또는 Delta 테이블로 저장합니다.

In [0]:
import json
import time
import os
import pandas as pd
from datetime import datetime, timedelta
from api_client import ATAPIClient

# API 키 설정
at_key = dbutils.secrets.get(scope="agrofood-api-prod", key="publicdata-service-key_yjs")

# 클라이언트 초기화
at_client = ATAPIClient(at_key)

### 1. 날짜 설정
- 기본값은 **오늘 날짜** 입니다.
- 특정 날짜를 수집하려면 `target_date`를 직접 변경하세요.
- Databricks Jobs에서는 widget 파라미터로 전달할 수 있습니다.

In [0]:
#  어제 날짜 자동 설정 (기본값)
target_date = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")

# 수동 지정 (테스트용, 필요시 주석 해제)
# target_date = "20260413"

print(f"수집 대상 날짜: {target_date}")

### 2. 품목코드 참조 데이터 불러오기
- `ref_sheet_품목코드.csv` 파일을 읽어 품목 리스트를 생성합니다.

In [0]:
# 품목코드 참조 파일 경로
ref_file_path = '/Workspace/Users/biod1614@gmail.com/ref_sheet_품목코드.csv'

try:
    df_ref = pd.read_csv(ref_file_path, encoding='utf-8-sig')
except:
    df_ref = pd.read_csv(ref_file_path, encoding='cp949')

print(f"[INFO] 품목코드 참조 데이터: {len(df_ref)}개 품목 로드 완료")
display(df_ref.head())

### 3. 일별 가격정보 수집
- `at_client.fetch_day_price()`를 사용하여 전체 품목의 당일 가격 데이터를 수집합니다.
- 기존 수집 코드와 동일한 페이지네이션 루프를 사용합니다.

In [0]:
print(f"{'='*60}")
print(f" 일별 가격정보 수집 시작")
print(f" 수집 시작 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

# 출력 파일 경로 (날짜별 파일)
output_dir = "/Volumes/bronze_api/agrofood_perday/volumn"
os.makedirs(output_dir, exist_ok=True)

all_collected_items = []
success_count = 0
no_data_count = 0

for index, row in df_ref.iterrows():
    ctgry_cd = str(row['부류코드'])
    item_cd = str(row['품목코드']).zfill(3)
    item_nm = row['품목명']
    
    print(f"\n[{index+1}/{len(df_ref)}] {item_nm} (부류:{ctgry_cd}, 품목:{item_cd}) 수집 중...")
    
    item_day_items = []
    page = 1
    rows_per_page = 100
    is_success = True
    
    while True:
        day_data = at_client.fetch_day_price(
            date_gte=target_date,
            date_lte=target_date,
            ctgry_cd=ctgry_cd,
            item_cd=item_cd,
            page=page,
            rows=rows_per_page,
            return_type="json"
        )
        
        # API 에러 처리
        if day_data is None:
            print(f" -> API 통신 에러 발생. 재시도...")
            time.sleep(2)
            day_data = at_client.fetch_day_price(
                date_gte=target_date,
                date_lte=target_date,
                ctgry_cd=ctgry_cd,
                item_cd=item_cd,
                page=page,
                rows=rows_per_page,
                return_type="json"
            )
            if day_data is None:
                print(f" -> 재시도 실패. 이 품목을 건너뜁니다.")
                is_success = False
                break
        
        body = day_data.get('response', {}).get('body', {})
        items = body.get('items', {}).get('item', [])
        total_count = body.get('totalCount', 0)
        
        if isinstance(items, dict):
            items = [items]
        
        if not items:
            break
        
        item_day_items.extend(items)
        
        if len(item_day_items) >= int(total_count):
            break
        
        page += 1
        time.sleep(0.05)  # API 부하 방지
    
    if item_day_items:
        all_collected_items.extend(item_day_items)
        success_count += 1
        print(f" -> {item_nm}: {len(item_day_items)}건 수집 완료")
    else:
        no_data_count += 1
        print(f" -> {item_nm}: 해당 날짜 데이터 없음")

print(f"\n{'='*60}")
print(f" 수집 완료 요약")
print(f" 대상 날짜: {target_date}")
print(f" 데이터 있는 품목: {success_count}개")
print(f" 데이터 없는 품목: {no_data_count}개")
print(f" 총 수집 건수: {len(all_collected_items)}건")
print(f" 완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

### 4. 수집 결과 저장
- **로컬/Jupyter**: CSV 파일로 저장 (`price_day_YYYYMMDD.csv`)
- **Databricks**: Delta 테이블로 저장 (아래 주석 해제하여 사용)
- 날짜pytz로시간대변경

In [0]:
import pytz
kst = pytz.timezone('Asia/Seoul')

if all_collected_items:
    df_result = pd.DataFrame(all_collected_items)
    
    df_result['collect_time'] = datetime.now(kst).strftime('%Y-%m-%d_%H:%M:%S')
    
    # ── CSV 저장 (로컬/Jupyter 환경) ──
    save_path = os.path.join(output_dir, f"일별가격정보_{target_date}.csv")
    df_result.to_csv(save_path, mode='w', encoding='utf-8', index=False)
    
    # 결과 미리보기
    print(f"수집된 일별 가격 데이터 미리보기 (총 {len(df_result)}건):")
    display(df_result.head(10))

else:
    print(f"[INFO] {target_date} 날짜에 수집된 데이터가 없습니다.")
    print("       (공휴일/주말 등 가격 조사가 없는 날일 수 있습니다)")

### 5. Delta 테이블 적재 (Append)
- 수집된 데이터를 `bronze_api.agrofood_perday.perday` Delta 테이블에 직접 적재합니다.
- **중복 방지**: 동일 날짜(`exmn_ymd`) 데이터를 먼저 삭제한 후 적재하여, 재실행 시에도 안전합니다.

In [0]:
if all_collected_items:
    # pandas → Spark DataFrame 변환 (모든 컬럼 STRING으로 통일)
    df_spark = spark.createDataFrame(df_result.astype(str))

    table_name = "bronze_api.agrofood_perday.perday"

    # 동일 날짜 데이터 중복 방지 (재실행 대비 DELETE → APPEND)
    spark.sql(f"DELETE FROM {table_name} WHERE exmn_ymd = '{target_date}'")
    print(f"[INFO] 기존 {target_date} 데이터 삭제 완료")

    # append 모드로 Delta 테이블에 적재
    df_spark.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)

    # 적재 확인
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name} WHERE exmn_ymd = '{target_date}'").first()['cnt']
    total = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").first()['cnt']
    print(f"[INFO] {table_name} 테이블에 {count}건 적재 완료 (날짜: {target_date})")
    print(f"[INFO] 테이블 전체 건수: {total:,}건")
else:
    print(f"[INFO] 적재할 데이터가 없습니다.")